# PGD Attack & Adversarial Training on CIFAR-10

This notebook demonstrates **PGD** attack and **adversarial training** on CIFAR-10 using `sleight`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sleight.data import get_dataset
from sleight.models import get_cifar10_cnn_model
from sleight.attacks import pgd_attack
from sleight.defenses import adversarial_train
from sleight.evaluation import evaluate_robustness

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = get_dataset("cifar10", one_hot=True)
y_train_int = np.argmax(y_train, axis=1)
y_test_int = np.argmax(y_test, axis=1)

In [ ]:
# Train standard model
model = get_cifar10_cnn_model()
model.fit(x_train, y_train, epochs=5, batch_size=64, verbose=1)

In [ ]:
# Evaluate under PGD
epsilon = 0.03
x_sub = x_test[:200]
y_sub_int = y_test_int[:200]

results_std = evaluate_robustness(model, x_sub, y_sub_int, pgd_attack, epsilon, alpha=0.005, num_iter=10)
print(f"Standard model - Clean: {results_std['clean_accuracy']:.4f}, Adv: {results_std['adversarial_accuracy']:.4f}")

In [ ]:
# Adversarial training
robust_model = get_cifar10_cnn_model()
robust_model = adversarial_train(
    robust_model, x_train[:5000], y_train_int[:5000],
    pgd_attack, epsilon=0.03, epochs=3, batch_size=64
)

In [ ]:
# Evaluate robust model
results_robust = evaluate_robustness(robust_model, x_sub, y_sub_int, pgd_attack, epsilon, alpha=0.005, num_iter=10)
print(f"Robust model - Clean: {results_robust['clean_accuracy']:.4f}, Adv: {results_robust['adversarial_accuracy']:.4f}")